# Model Evaluation
Load saved models and visualize performance on the test set.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import joblib
import polars as pl
from sklearn.metrics import precision_score, recall_score

from common.engine import load_and_prepare_data
from common.visualizations import (
    plot_confusion_matrix,
    plot_cross_validation_scores,
    plot_metrics_comparison,
    plot_mlp_loss_curve,
    plot_optuna_trials,
)

/Users/vincent/Labs/cancer-detection/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data and models

In [2]:
ROOT = Path.cwd().parent

features_train, features_test, target_train, target_test, _ = load_and_prepare_data()

decision_tree = joblib.load(ROOT / 'models/decision_tree.joblib')
knn           = joblib.load(ROOT / 'models/knn.joblib')
mlp           = joblib.load(ROOT / 'models/mlp.joblib')

tree_predictions = decision_tree.predict(features_test)
knn_predictions  = knn.predict(features_test)
mlp_predictions  = mlp.predict(features_test)

## Decision Tree

In [3]:
from sklearn import model_selection

cross_validation_scores = model_selection.cross_validate(
    decision_tree, features_train, target_train,
    cv=5, scoring=['accuracy'], return_train_score=True
)
plot_cross_validation_scores(cross_validation_scores, 'Decision Tree').show()
plot_confusion_matrix(target_test, tree_predictions, 'Decision Tree').show()

## KNN

In [4]:
import optuna
from sklearn import neighbors

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    k = trial.suggest_int('n_neighbors', 2, 41)
    model = neighbors.KNeighborsClassifier(n_neighbors=k)
    return model_selection.cross_val_score(model, features_train, target_train, cv=5, scoring='accuracy').mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40)

plot_optuna_trials(study, 'KNN').show()
plot_confusion_matrix(target_test, knn_predictions, 'KNN').show()

## MLP

In [5]:
plot_mlp_loss_curve(mlp, 'MLP').show()
plot_confusion_matrix(target_test, mlp_predictions, 'MLP').show()

## Model Comparison

In [6]:
models = ['Decision Tree', 'KNN', 'MLP']
all_predictions = [tree_predictions, knn_predictions, mlp_predictions]

comparison = pl.DataFrame({
    'Model':     models,
    'Accuracy':  [precision_score(target_test, p, average='weighted') for p in all_predictions],
    'Precision': [precision_score(target_test, p, average='weighted') for p in all_predictions],
    'Recall':    [recall_score(target_test, p, average='weighted')    for p in all_predictions],
})

print(comparison)

plot_metrics_comparison(
    comparison['Model'].to_list(),
    comparison['Accuracy'].to_list(),
    comparison['Precision'].to_list(),
    comparison['Recall'].to_list(),
).show()

shape: (3, 4)
┌───────────────┬──────────┬───────────┬──────────┐
│ Model         ┆ Accuracy ┆ Precision ┆ Recall   │
│ ---           ┆ ---      ┆ ---       ┆ ---      │
│ str           ┆ f64      ┆ f64       ┆ f64      │
╞═══════════════╪══════════╪═══════════╪══════════╡
│ Decision Tree ┆ 0.947368 ┆ 0.947368  ┆ 0.947368 │
│ KNN           ┆ 0.947368 ┆ 0.947368  ┆ 0.947368 │
│ MLP           ┆ 0.982456 ┆ 0.982456  ┆ 0.982456 │
└───────────────┴──────────┴───────────┴──────────┘
